In [0]:
# ==========================================================
# Project : Lyon Rental Market Analytics
# Layer   : Bronze
# Notebook: 02_ingest_dvf_bronze
#
# Description
# ----------------------------------------------------------
# This notebook ingests the dvf dataset 
# from Azure Data Lake Storage Gen2 into the Bronze layer.
#
# No transformation is performed except adding ingestion metadata.
#
# Input
# ----------------------------------------------------------
# CSV file stored in ADLS Gen2
#
# Output
# ----------------------------------------------------------
# Bronze Delta Table
#
# Author : Xin Yao
# ==========================================================


# ---------------------------------------
# 1. Import libraries
# ---------------------------------------

from pyspark.sql.functions import (
    current_timestamp,
    input_file_name,
    lit,
    col
)

# ---------------------------------------
# 2. Configuration
# ---------------------------------------

STORAGE_ACCOUNT = "stlyonloyersdev"
CONTAINER = "lyon-data"

SOURCE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "bronze/dvf/dvf_plus_d69.csv"
)

TARGET_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "bronze/delta/dvf"
)

# ---------------------------------------
# 3. Read raw CSV
# ---------------------------------------

print("Reading raw CSV...")

df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .option("delimiter", "|")     
         .csv(SOURCE_PATH)
)

display(df.limit(5))

# ---------------------------------------
# 4. Add ingestion metadata
# ---------------------------------------

print("Adding metadata...")

bronze_df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("data_source", lit("dvf"))
)

display(bronze_df.limit(5))

# ---------------------------------------
# 5. Save as Bronze Delta Table
# ---------------------------------------

print("Writing Bronze Delta Table...")

(
    bronze_df.write
             .format("delta")
             .mode("overwrite")
             .save(TARGET_PATH)
)

print("Bronze ingestion completed.")

# ---------------------------------------
# 6. Validation
# ---------------------------------------

validation_df = spark.read.format("delta").load(TARGET_PATH)

print(f"Number of records : {validation_df.count()}")

display(validation_df.limit(5))